In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("zillow_median.csv")

# Preview data
df.head()

,RegionID,RegionName,City,State,Metro,CountyName,SizeRank,1996-04,1996-05,1996-06,...,2016-12,2017-01,2017-02,2017-03,2017-04,2017-05,2017-06,2017-07,2017-08,2017-09
0,274772,Northeast Dallas,Dallas,TX,Dallas-Fort Worth,Dallas,1,NaN,NaN,NaN,...,277500,278300,281600,285900,289900,292900,292900,291800,290900,290100
1,192689,Paradise,Las Vegas,NV,Las Vegas,Clark,2,115400.0,115300.0,115500.0,...,196500,199100,201400,202900,204200,206300,208900,211100,213400,215500
2,112345,Maryvale,Phoenix,AZ,Phoenix,Maricopa,3,58900.0,58900.0,58900.0,...,147500,148300,148600,148900,149500,150800,152500,153900,155100,156100
3,27080,Sherman Oaks,Los Angeles,CA,Los Angeles-Long Beach-Anaheim,Los Angeles,4,216200.0,218200.0,218900.0,...,869700,872500,876200,879500,882400,886400,891100,896200,903200,910200
4,118208,South Los Angeles,Los Angeles,CA,Los Angeles-Long Beach-Anaheim,Los Angeles,5,117600.0,118700.0,119600.0,...,401900,406100,411000,417300,423400,427100,429700,432500,435300,437500


In [3]:
print(df.shape)
print(df.info())
print(df.describe())

# Check missing values
print(df.isnull().sum())

(5802, 265)
<class 'pandas.DataFrame'>
RangeIndex: 5802 entries, 0 to 5801
Columns: 265 entries, RegionID to 2017-09
dtypes: float64(219), int64(41), str(5)
memory usage: 12.0 MB
None
            RegionID     SizeRank       1996-04       1996-05       1996-06  \
count    5802.000000  5802.000000  4.200000e+03  4.249000e+03  4.250000e+03   
mean   287406.618752  2901.500000  1.265352e+05  1.266244e+05  1.267355e+05   
std    125890.140512  1675.037462  9.222263e+04  9.177551e+04  9.183236e+04   
min      3736.000000     1.000000  2.830000e+04  2.800000e+04  2.770000e+04   
25%    268485.750000  1451.250000  7.170000e+04  7.230000e+04  7.240000e+04   
50%    273228.500000  2901.500000  1.059000e+05  1.064000e+05  1.066000e+05   
75%    276610.750000  4351.750000  1.544000e+05  1.538000e+05  1.538750e+05   
max    753841.000000  5802.000000  2.636200e+06  2.636200e+06  2.635600e+06   

            1996-07       1996-08       1996-09       1996-10       1996-11  \
count  4.250000e+03  4.25

In [4]:
# Drop rows with missing values (basic approach)
df = df.dropna()

df.head()

,RegionID,RegionName,City,State,Metro,CountyName,SizeRank,1996-04,1996-05,1996-06,...,2016-12,2017-01,2017-02,2017-03,2017-04,2017-05,2017-06,2017-07,2017-08,2017-09
1,192689,Paradise,Las Vegas,NV,Las Vegas,Clark,2,115400.0,115300.0,115500.0,...,196500,199100,201400,202900,204200,206300,208900,211100,213400,215500
2,112345,Maryvale,Phoenix,AZ,Phoenix,Maricopa,3,58900.0,58900.0,58900.0,...,147500,148300,148600,148900,149500,150800,152500,153900,155100,156100
3,27080,Sherman Oaks,Los Angeles,CA,Los Angeles-Long Beach-Anaheim,Los Angeles,4,216200.0,218200.0,218900.0,...,869700,872500,876200,879500,882400,886400,891100,896200,903200,910200
4,118208,South Los Angeles,Los Angeles,CA,Los Angeles-Long Beach-Anaheim,Los Angeles,5,117600.0,118700.0,119600.0,...,401900,406100,411000,417300,423400,427100,429700,432500,435300,437500
5,192820,Sunrise Manor,Las Vegas,NV,Las Vegas,Clark,6,98200.0,98300.0,98200.0,...,162700,164100,165300,166200,167700,170000,172100,173400,174800,176500


In [7]:
df.info()

<class 'pandas.DataFrame'>
Index: 4105 entries, 1 to 5801
Columns: 265 entries, RegionID to 2017-09
dtypes: float64(219), int64(41), str(5)
memory usage: 8.5 MB


In [8]:
df.head()

,RegionID,RegionName,City,State,Metro,CountyName,SizeRank,1996-04,1996-05,1996-06,...,2016-12,2017-01,2017-02,2017-03,2017-04,2017-05,2017-06,2017-07,2017-08,2017-09
1,192689,Paradise,Las Vegas,NV,Las Vegas,Clark,2,115400.0,115300.0,115500.0,...,196500,199100,201400,202900,204200,206300,208900,211100,213400,215500
2,112345,Maryvale,Phoenix,AZ,Phoenix,Maricopa,3,58900.0,58900.0,58900.0,...,147500,148300,148600,148900,149500,150800,152500,153900,155100,156100
3,27080,Sherman Oaks,Los Angeles,CA,Los Angeles-Long Beach-Anaheim,Los Angeles,4,216200.0,218200.0,218900.0,...,869700,872500,876200,879500,882400,886400,891100,896200,903200,910200
4,118208,South Los Angeles,Los Angeles,CA,Los Angeles-Long Beach-Anaheim,Los Angeles,5,117600.0,118700.0,119600.0,...,401900,406100,411000,417300,423400,427100,429700,432500,435300,437500
5,192820,Sunrise Manor,Las Vegas,NV,Las Vegas,Clark,6,98200.0,98300.0,98200.0,...,162700,164100,165300,166200,167700,170000,172100,173400,174800,176500


In [13]:
df.columns


Index(['RegionID', 'RegionName', 'City', 'State', 'Metro', 'CountyName',
       'SizeRank', '1996-04', '1996-05', '1996-06',
       ...
       '2016-12', '2017-01', '2017-02', '2017-03', '2017-04', '2017-05',
       '2017-06', '2017-07', '2017-08', '2017-09'],
      dtype='str', length=265)

In [16]:
print(df.columns)

Index(['RegionID', 'RegionName', 'City', 'State', 'Metro', 'CountyName',
       'SizeRank', '1996-04', '1996-05', '1996-06',
       ...
       '2016-12', '2017-01', '2017-02', '2017-03', '2017-04', '2017-05',
       '2017-06', '2017-07', '2017-08', '2017-09'],
      dtype='str', length=265)
